In [2]:
!pip install -q kagglehub gradio
from google.colab import files
import pandas as pd

url = "https://raw.githubusercontent.com/Abdulrahman-Hussein0/Ardna/refs/heads/main/egypt_food_balance.csv"
df = pd.read_csv(url)
df.head()


,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (FBS),Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
0,FBS,Food Balances (2010-),818,Egypt,5511,Production,S2511,Wheat and products,2010,2010,1000 t,7185,I,Value imputed by a receiving agency,NaN
1,FBS,Food Balances (2010-),818,Egypt,5611,Import quantity,S2511,Wheat and products,2010,2010,1000 t,10825,I,Value imputed by a receiving agency,NaN
2,FBS,Food Balances (2010-),818,Egypt,5911,Export quantity,S2511,Wheat and products,2010,2010,1000 t,93,I,Value imputed by a receiving agency,NaN
3,FBS,Food Balances (2010-),818,Egypt,5301,Domestic supply quantity,S2511,Wheat and products,2010,2010,1000 t,18042,I,Value imputed by a receiving agency,NaN
4,FBS,Food Balances (2010-),818,Egypt,5511,Production,S2511,Wheat and products,2011,2011,1000 t,8410,I,Value imputed by a receiving agency,NaN


In [7]:
import glob, os
import pandas as pd
import kagglehub

fa_url = "https://raw.githubusercontent.com/Abdulrahman-Hussein0/Ardna/refs/heads/main/egypt_food_balance.csv"
fb = pd.read_csv(fa_url)

kpath = kagglehub.dataset_download("atharvaingle/crop-recommendation-dataset")
cr = pd.read_csv(os.path.join(kpath, "Crop_recommendation.csv"))

print("ملف FAOSTAT:", fa_url)
print("FAOSTAT:", fb.shape, "| Kaggle:", cr.shape)
print(fb["Item"].unique())

Using Colab cache for faster access to the 'crop-recommendation-dataset' dataset.
ملف FAOSTAT: https://raw.githubusercontent.com/Abdulrahman-Hussein0/Ardna/refs/heads/main/egypt_food_balance.csv
FAOSTAT: (865, 15) | Kaggle: (2200, 8)
['Wheat and products' 'Rice and products' 'Maize and products'
 'Potatoes and products' 'Sweet potatoes' 'Sugar cane' 'Beans'
 'Tomatoes and products' 'Onions' 'Vegetables, other'
 'Oranges, Mandarines' 'Lemons, Limes and products' 'Bananas'
 'Apples and products' 'Fruits, other' 'Pepper']


In [8]:
t = fb.pivot_table(index=["Item", "Year"], columns="Element", values="Value")
t["deficit"] = (1 - t["Production"] / t["Domestic supply quantity"]).clip(0, 1)

d = t.reset_index().query("Year >= 2021").groupby("Item")["deficit"].mean()
deficit = d.rename(lambda s: s.split(" and")[0].lower())

print(deficit.round(2).sort_values(ascending=False))

Item
maize                  0.57
wheat                  0.44
apples                 0.16
rice                   0.09
fruits, other          0.00
beans                  0.00
bananas                0.00
onions                 0.00
lemons, limes          0.00
potatoes               0.00
oranges, mandarines    0.00
sugar cane             0.00
sweet potatoes         0.00
tomatoes               0.00
vegetables, other      0.00
pepper                  NaN
Name: deficit, dtype: float64


In [9]:
# (حرارة min, حرارة max, pH min, pH max)
# أرقام تقريبية، هنراجعها بعدين من FAO EcoCrop
REQ = {
    "wheat":               (10, 24, 6.0, 7.5),
    "maize":               (18, 32, 5.8, 7.5),
    "rice":                (20, 35, 5.5, 7.0),
    "apples":              (6, 22, 5.5, 7.0),
    "potatoes":            (12, 22, 5.0, 6.5),
    "tomatoes":            (18, 28, 6.0, 7.0),
    "onions":              (12, 24, 6.0, 7.0),
    "oranges, mandarines": (15, 30, 6.0, 7.5),
}

# كل محصول بيتزرع في أنهي موسم؟
SEASON = {
    "wheat": "winter", "potatoes": "winter", "tomatoes": "winter",
    "onions": "winter", "apples": "winter",
    "maize": "summer", "rice": "summer", "oranges, mandarines": "summer",
}

def range_score(x, lo, hi, tol):
    if lo <= x <= hi:
        return 1.0
    dist = (lo - x) if x < lo else (x - hi)
    return max(0.0, 1 - dist / tol)

def suitability(crop, temp, ph):
    tmin, tmax, pmin, pmax = REQ[crop]
    return range_score(temp, tmin, tmax, 10) * range_score(ph, pmin, pmax, 1.5)

# اختبار سريع
print(suitability("wheat", 15, 7.8))
print(suitability("rice", 32, 7.8))

0.8000000000000002
0.4666666666666668


In [10]:
# أرقام مثال، هنبدلها بأرقام حقيقية بعدين
REGIONS = {
    "Minya":   {"winter": 15, "summer": 32, "ph": 7.8},
    "Beheira": {"winter": 14, "summer": 29, "ph": 7.9},
}

def recommend(gov):
    r, rows = REGIONS[gov], []
    for crop in REQ:
        suit = suitability(crop, r[SEASON[crop]], r["ph"])
        dfc = float(deficit[crop])
        rows.append((crop, round(dfc, 2), round(suit, 2), round(dfc * suit, 2)))
    return (pd.DataFrame(rows, columns=["crop", "deficit", "suitability", "score"])
              .sort_values("score", ascending=False).reset_index(drop=True))

print(recommend("Minya"))
print()
print(recommend("Beheira"))

                  crop  deficit  suitability  score
0                maize     0.57         0.80   0.45
1                wheat     0.44         0.80   0.35
2               apples     0.16         0.47   0.07
3                 rice     0.09         0.47   0.04
4             potatoes     0.00         0.13   0.00
5             tomatoes     0.00         0.33   0.00
6               onions     0.00         0.47   0.00
7  oranges, mandarines     0.00         0.64   0.00

                  crop  deficit  suitability  score
0                maize     0.57         0.73   0.41
1                wheat     0.44         0.73   0.32
2               apples     0.16         0.40   0.06
3                 rice     0.09         0.40   0.03
4             potatoes     0.00         0.07   0.00
5             tomatoes     0.00         0.24   0.00
6               onions     0.00         0.40   0.00
7  oranges, mandarines     0.00         0.73   0.00


In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

FEATURES = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
Xtr, Xte, ytr, yte = train_test_split(cr[FEATURES], cr["label"],
                                      test_size=0.2, random_state=42)
rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(Xtr, ytr)
print("Accuracy:", round(rf.score(Xte, yte), 3))

Accuracy: 0.993


In [12]:
KAGGLE_NAME = {"maize": "maize", "rice": "rice",
               "apples": "apple", "oranges, mandarines": "orange"}
EXTRA = {"N": 50, "P": 53, "K": 48, "humidity": 65, "water": 100}

def rf_suitability(crop, row):
    if crop not in KAGGLE_NAME:
        return None
    probs = pd.Series(rf.predict_proba(pd.DataFrame([row], columns=FEATURES))[0],
                      index=rf.classes_)
    return float(probs[KAGGLE_NAME[crop]] / probs.max())

def recommend_ml(gov):
    r, rows = REGIONS[gov], []
    for crop in REQ:
        temp = r[SEASON[crop]]
        s_rule = suitability(crop, temp, r["ph"])
        row = [EXTRA["N"], EXTRA["P"], EXTRA["K"], temp,
               EXTRA["humidity"], r["ph"], EXTRA["water"]]
        s_rf = rf_suitability(crop, row)
        suit = s_rule if s_rf is None else 0.5 * s_rule + 0.5 * s_rf
        dfc = float(deficit[crop])
        rows.append((crop, round(dfc, 2), round(s_rule, 2),
                     None if s_rf is None else round(s_rf, 2),
                     round(suit, 2), round(dfc * suit, 2)))
    cols = ["crop", "deficit", "rule", "ml", "suitability", "score"]
    return (pd.DataFrame(rows, columns=cols)
              .sort_values("score", ascending=False).reset_index(drop=True))

print(recommend_ml("Minya"))

                  crop  deficit  rule    ml  suitability  score
0                wheat     0.44  0.80   NaN         0.80   0.35
1                maize     0.57  0.80  0.02         0.41   0.23
2               apples     0.16  0.47  0.00         0.23   0.04
3                 rice     0.09  0.47  0.04         0.25   0.02
4             potatoes     0.00  0.13   NaN         0.13   0.00
5             tomatoes     0.00  0.33   NaN         0.33   0.00
6               onions     0.00  0.47   NaN         0.47   0.00
7  oranges, mandarines     0.00  0.64  0.02         0.33   0.00


In [13]:
print(cr.groupby("label")[["temperature", "humidity", "rainfall"]]
        .agg(["min", "max"]).loc[["maize", "rice", "apple", "orange"]].round(0))

       temperature       humidity       rainfall       
               min   max      min   max      min    max
label                                                  
maize         18.0  27.0     55.0  75.0     61.0  110.0
rice          20.0  27.0     80.0  85.0    183.0  299.0
apple         21.0  24.0     90.0  95.0    100.0  125.0
orange        10.0  35.0     90.0  95.0    100.0  120.0


In [14]:
def recommend_final(gov):
    df = recommend_ml(gov).copy()
    df["suitability"] = df["rule"]
    df["score"] = (df["deficit"] * df["suitability"]).round(2)
    df["ml_agrees"] = df["ml"].apply(
        lambda x: "" if pd.isna(x) else ("✅" if x >= 0.5 else "⚠️"))
    cols = ["crop", "deficit", "suitability", "score", "ml", "ml_agrees"]
    return df.sort_values("score", ascending=False).reset_index(drop=True)[cols]

print(recommend_final("Minya"))

                  crop  deficit  suitability  score    ml ml_agrees
0                maize     0.57         0.80   0.46  0.02        ⚠️
1                wheat     0.44         0.80   0.35   NaN          
2               apples     0.16         0.47   0.08  0.00        ⚠️
3                 rice     0.09         0.47   0.04  0.04        ⚠️
4             potatoes     0.00         0.13   0.00   NaN          
5             tomatoes     0.00         0.33   0.00   NaN          
6               onions     0.00         0.47   0.00   NaN          
7  oranges, mandarines     0.00         0.64   0.00  0.02        ⚠️


In [15]:
# 1) إيه المدى اللي Kaggle شايفه مناسب للمحاصيل بتاعتنا؟
print(cr.groupby("label")[FEATURES].mean().loc[["maize", "rice", "apple", "orange"]].round(0))
print()

# 2) الموديل شايف مدخلات المنيا (صيف) أقرب لأنهي محصول؟
row = [50, 53, 48, 32, 65, 7.8, 100]
probs = pd.Series(rf.predict_proba(pd.DataFrame([row], columns=FEATURES))[0], index=rf.classes_)
print(probs.sort_values(ascending=False).head(3).round(2))

           N      P      K  temperature  humidity   ph  rainfall
label                                                           
maize   78.0   48.0   20.0         22.0      65.0  6.0      85.0
rice    80.0   48.0   40.0         24.0      82.0  6.0     236.0
apple   21.0  134.0  200.0         23.0      92.0  6.0     113.0
orange  20.0   17.0   10.0         23.0      92.0  7.0     110.0

mango         0.27
papaya        0.22
pigeonpeas    0.14
dtype: float64


In [16]:
base = cr[cr.label == "maize"][FEATURES].mean()   # صف مثالي للذرة

def p_maize(**changes):
    r = base.copy()
    for k, v in changes.items():
        r[k] = v
    p = rf.predict_proba(pd.DataFrame([r], columns=FEATURES))[0]
    return round(dict(zip(rf.classes_, p))["maize"], 2)

print("الذرة المثالية:", p_maize())
print("حرارة 32      :", p_maize(temperature=32))
print("pH 7.8        :", p_maize(ph=7.8))
print("K = 48        :", p_maize(K=48))
print("كلهم          :", p_maize(temperature=32, ph=7.8, K=48, N=50, P=53))

الذرة المثالية: 1.0
حرارة 32      : 0.59
pH 7.8        : 0.78
K = 48        : 0.38
كلهم          : 0.0


In [17]:
F3 = ["temperature", "humidity", "rainfall"]
Xtr3, Xte3, ytr3, yte3 = train_test_split(cr[F3], cr["label"],
                                          test_size=0.2, random_state=42)
rf3 = RandomForestClassifier(n_estimators=200, random_state=42).fit(Xtr3, ytr3)
print("Accuracy:", round(rf3.score(Xte3, yte3), 3))

def show(temp, hum, water):
    p = pd.Series(rf3.predict_proba(pd.DataFrame([[temp, hum, water]], columns=F3))[0],
                  index=rf3.classes_)
    print(p[["maize", "rice", "apple", "orange"]].round(2).to_dict(),
          "| الأعلى:", p.idxmax(), round(p.max(), 2))

show(22, 65, 85)    # ذرة مثالية من Kaggle
show(28, 50, 100)   # صيف المنيا (متوسط الموسم، مش الذروة)

Accuracy: 0.907
{'maize': 0.94, 'rice': 0.0, 'apple': 0.0, 'orange': 0.0} | الأعلى: maize 0.94
{'maize': 0.0, 'rice': 0.0, 'apple': 0.0, 'orange': 0.0} | الأعلى: mango 0.86


In [18]:
REGIONS["Minya"].update({"summer": 28, "humidity": 50, "water": 100})
REGIONS["Beheira"].update({"summer": 27, "humidity": 65, "water": 120})

def recommend_final(gov):
    r, rows = REGIONS[gov], []
    for crop in REQ:
        temp = r[SEASON[crop]]
        suit = suitability(crop, temp, r["ph"])
        dfc = float(deficit[crop])
        p = None
        if crop in KAGGLE_NAME:
            probs = pd.Series(
                rf3.predict_proba(pd.DataFrame([[temp, r["humidity"], r["water"]]],
                                               columns=F3))[0],
                index=rf3.classes_)
            p = round(float(probs[KAGGLE_NAME[crop]]), 2)
        rows.append((crop, round(dfc, 2), round(suit, 2), round(dfc * suit, 2), p))
    cols = ["crop", "deficit", "suitability", "score", "ml_opinion"]
    return (pd.DataFrame(rows, columns=cols)
              .sort_values("score", ascending=False).reset_index(drop=True))

print(recommend_final("Minya"))
print()
print(recommend_final("Beheira"))

                  crop  deficit  suitability  score  ml_opinion
0                maize     0.57         0.80   0.45         0.0
1                wheat     0.44         0.80   0.35         NaN
2               apples     0.16         0.47   0.07         0.0
3                 rice     0.09         0.47   0.04         0.0
4             potatoes     0.00         0.13   0.00         NaN
5             tomatoes     0.00         0.33   0.00         NaN
6               onions     0.00         0.47   0.00         NaN
7  oranges, mandarines     0.00         0.80   0.00         0.0

                  crop  deficit  suitability  score  ml_opinion
0                maize     0.57         0.73   0.41        0.04
1                wheat     0.44         0.73   0.32         NaN
2               apples     0.16         0.40   0.06        0.01
3                 rice     0.09         0.40   0.03        0.00
4             potatoes     0.00         0.07   0.00         NaN
5             tomatoes     0.00        

In [19]:
from sklearn.linear_model import LinearRegression
import numpy as np

ts = t["deficit"].reset_index()
ts["crop"] = ts["Item"].apply(lambda s: s.split(" and")[0].lower())

rows = []
for crop in REQ:
    d = ts[ts["crop"] == crop].sort_values("Year")
    X, y = d[["Year"]].values, d["deficit"].values
    tr = (d["Year"] <= 2020).values
    te = (d["Year"] > 2020).values
    m = LinearRegression().fit(X[tr], y[tr])
    err_model = np.abs(m.predict(X[te]) - y[te]).mean()
    err_base = np.abs(y[tr].mean() - y[te]).mean()
    m2 = LinearRegression().fit(X, y)
    f2026 = float(np.clip(m2.predict([[2026]])[0], 0, 1))
    rows.append((crop, round(y[-1], 2), round(f2026, 2),
                 round(err_model, 3), round(err_base, 3)))

print(pd.DataFrame(rows, columns=["crop", "deficit_2023", "forecast_2026",
                                  "err_model", "err_baseline"]))

                  crop  deficit_2023  forecast_2026  err_model  err_baseline
0                wheat          0.46           0.42      0.068         0.106
1                maize          0.59           0.60      0.030         0.036
2                 rice          0.07           0.13      0.066         0.029
3               apples          0.12           0.06      0.026         0.137
4             potatoes          0.00           0.00      0.000         0.000
5             tomatoes          0.00           0.00      0.000         0.000
6               onions          0.00           0.00      0.000         0.000
7  oranges, mandarines          0.00           0.00      0.000         0.000


In [20]:
import json

fc = pd.DataFrame(rows, columns=["crop", "deficit_2023", "forecast_2026",
                                 "err_model", "err_baseline"]).set_index("crop")
fc["use_forecast"] = fc["err_model"] < fc["err_baseline"]

deficit_used = {c: (float(fc.loc[c, "forecast_2026"]) if fc.loc[c, "use_forecast"]
                    else float(deficit[c])) for c in REQ}

def recommend_v2(gov):
    r, out = REGIONS[gov], []
    for crop in REQ:
        suit = suitability(crop, r[SEASON[crop]], r["ph"])
        dfc = deficit_used[crop]
        out.append({"crop": crop, "deficit": round(dfc, 2),
                    "suitability": round(suit, 2), "score": round(dfc * suit, 2)})
    return pd.DataFrame(out).sort_values("score", ascending=False).reset_index(drop=True)

print(recommend_v2("Minya"))

                  crop  deficit  suitability  score
0                maize     0.60         0.80   0.48
1                wheat     0.42         0.80   0.34
2                 rice     0.09         0.47   0.04
3               apples     0.06         0.47   0.03
4             potatoes     0.00         0.13   0.00
5             tomatoes     0.00         0.33   0.00
6               onions     0.00         0.47   0.00
7  oranges, mandarines     0.00         0.80   0.00


In [21]:
import re
import pandas as pd
import gradio as gr

# ---------- الفايض (حساب داخلي) ----------
_s = (t["Production"] / t["Domestic supply quantity"] - 1).clip(lower=0)
surplus = (_s.reset_index(name="surplus").query("Year >= 2021")
             .groupby("Item")["surplus"].mean()
             .rename(lambda x: x.split(" and")[0].lower()))

AR = {"maize": "ذرة", "wheat": "قمح", "rice": "أرز", "apples": "تفاح",
      "potatoes": "بطاطس", "tomatoes": "طماطم", "onions": "بصل",
      "oranges, mandarines": "برتقال ويوسفي"}

GOV_AR = {"Minya": "المنيا", "Beheira": "البحيرة", "Menoufia": "المنوفية",
          "Sharqia": "الشرقية", "Aswan": "أسوان", "Assiut": "أسيوط"}

# ---------- بيانات المحافظات ----------
# ⚠️ أرقام تقريبية من عندي مش من مصدر. بدّلها بأرقام حقيقية قبل العرض:
# winter/summer = متوسط حرارة الموسم (NASA POWER)، ph و ec = حموضة وملوحة التربة (dS/m)
REGIONS = {
    "Minya":    {"winter": 15, "summer": 29, "ph": 7.8, "ec": 2.0},
    "Beheira":  {"winter": 14, "summer": 27, "ph": 8.0, "ec": 4.0},
    "Menoufia": {"winter": 14, "summer": 28, "ph": 7.8, "ec": 2.0},
    "Sharqia":  {"winter": 14, "summer": 28, "ph": 7.9, "ec": 3.0},
    "Aswan":    {"winter": 18, "summer": 34, "ph": 7.7, "ec": 2.5},
    "Assiut":   {"winter": 16, "summer": 31, "ph": 7.8, "ec": 1.8},
}

# الأرز مسموح في محافظات محددة بقرار وزارة الري كل سنة (راجع قرار 32 لسنة 2026)
RICE_OK = {"Beheira", "Sharqia"}

# ---------- متطلبات المحاصيل (تقريبية) ----------
# t = (أقل/أعلى حرارة مقبولة) و (المدى المثالي)، ph بنفس الفكرة
# ec = (حد الملوحة اللي بعده الإنتاج يقل، نسبة النقص % لكل وحدة ملوحة زيادة)
CROP = {
    "wheat":    dict(t_abs=(3, 28),  t_opt=(10, 17), ph_abs=(5, 9),   ph_opt=(6.0, 8.0), ec=(6.0, 7.1)),
    "maize":    dict(t_abs=(10, 38), t_opt=(20, 30), ph_abs=(5, 9),   ph_opt=(5.8, 8.0), ec=(1.7, 12)),
    "rice":     dict(t_abs=(12, 40), t_opt=(22, 32), ph_abs=(4.5, 9), ph_opt=(5.5, 8.0), ec=(3.0, 12)),
    "potatoes": dict(t_abs=(4, 28),  t_opt=(12, 20), ph_abs=(4, 8.5), ph_opt=(5.0, 7.5), ec=(1.7, 12)),
    "tomatoes": dict(t_abs=(8, 35),  t_opt=(18, 27), ph_abs=(5, 8.5), ph_opt=(6.0, 7.8), ec=(2.5, 9.9)),
    "onions":   dict(t_abs=(5, 32),  t_opt=(12, 24), ph_abs=(5, 8.5), ph_opt=(6.0, 7.8), ec=(1.2, 16)),
    "apples":   dict(t_abs=(-2, 30), t_opt=(6, 20),  ph_abs=(4.5, 8.5), ph_opt=(5.5, 7.5), ec=(1.7, 16)),
    "oranges, mandarines": dict(t_abs=(8, 40), t_opt=(15, 30), ph_abs=(5, 9), ph_opt=(6.0, 8.0), ec=(1.7, 16)),
}

# ---------- اقتصاديات المحاصيل (للفدان في الموسم) ----------
# ⚠️ كلها تقديرات مني مش من مصدر. حدّثها من أسعار السوق الحالية ومن مزارعين:
# y = الإنتاجية بالطن، price = السعر للطن بالجنيه (أقل، أعلى)، cost = التكلفة بالجنيه
ECON = {
    "wheat":    dict(y=(2.6, 3.0),  price=(14500, 16000), cost=25000),
    "maize":    dict(y=(3.0, 3.6),  price=(11000, 13000), cost=24000),
    "rice":     dict(y=(3.4, 4.0),  price=(13000, 15000), cost=30000),
    "potatoes": dict(y=(10, 12),    price=(8000, 10000),  cost=60000),
    "tomatoes": dict(y=(16, 20),    price=(5500, 8000),   cost=65000),
    "onions":   dict(y=(12, 16),    price=(5000, 7000),   cost=45000),
}

# راجعها من FAOSTAT Trade أو TradeMap قبل العرض
EXPORT_TO = {
    "oranges, mandarines": ["روسيا", "السعودية", "هولندا", "الإمارات"],
    "potatoes": ["روسيا", "هولندا", "السعودية", "بريطانيا"],
    "onions": ["السعودية", "هولندا", "ماليزيا", "الإمارات"],
    "tomatoes": ["السعودية", "الإمارات"],
}

MIN_FIT = 0.4
WEAK = {"temp": "درجة الحرارة", "ph": "حموضة التربة", "salt": "ملوحة التربة"}

# ---------- حساب الملاءمة ----------
def trap(x, abs_rng, opt_rng):
    a_lo, a_hi = abs_rng
    o_lo, o_hi = opt_rng
    if o_lo <= x <= o_hi:
        return 1.0
    if x < o_lo:
        return max(0.0, (x - a_lo) / (o_lo - a_lo))
    return max(0.0, (a_hi - x) / (a_hi - o_hi))

def salt_factor(ec, thr, slope):
    return max(0.0, 1 - slope / 100 * max(0.0, ec - thr))

def suit_parts(crop, r):
    c = CROP[crop]
    return {"temp": trap(r[SEASON[crop]], c["t_abs"], c["t_opt"]),
            "ph": trap(r["ph"], c["ph_abs"], c["ph_opt"]),
            "salt": salt_factor(r["ec"], *c["ec"])}

def recommend_v3(gov):
    r, rows = REGIONS[gov], []
    for crop in CROP:
        parts = suit_parts(crop, r)
        s = parts["temp"] * parts["ph"] * parts["salt"]
        if crop == "rice" and gov not in RICE_OK:
            s = 0.0
        d = deficit_used[crop]
        worst = min(parts, key=parts.get)
        weak = WEAK[worst] if parts[worst] < 0.9 else ""
        rows.append({"crop": crop, "deficit": round(d, 2),
                     "suitability": round(s, 2), "score": round(d * s, 2),
                     "weak": weak})
    return (pd.DataFrame(rows)
              .sort_values("score", ascending=False).reset_index(drop=True))

def profit_range(crop, suit):
    e = ECON[crop]
    y = sum(e["y"]) / 2 * suit          # الإنتاجية بتقل لو الملاءمة قليلة
    return (y * e["price"][0] - e["cost"], y * e["price"][1] - e["cost"])

# ---------- أدوات التنسيق ----------
def p(x):
    return f"<bdi><b>{x * 100:.0f}%</b></bdi>"

def num(x):
    return f"<bdi><b>{x:g}</b></bdi>"

def money(lo, hi):
    top = max(abs(lo), abs(hi))
    if top >= 1_000_000:
        f, unit = (lambda x: f"{x / 1_000_000:.1f}"), "مليون جنيه"
    elif top >= 1_000:
        f, unit = (lambda x: f"{round(x / 1_000):,}"), "ألف جنيه"
    else:
        f, unit = (lambda x: f"{round(x):,}"), "جنيه"
    return f"من <bdi><b>{f(lo)}</b></bdi> إلى <bdi><b>{f(hi)}</b></bdi> {unit}"

def fit_word(x):
    return "ممتاز" if x >= 0.8 else "كويس" if x >= 0.5 else "مقبول"

def card(*lines):
    body = "".join(f"<div style='margin:4px 0'>{l}</div>" for l in lines)
    return ("<div style='padding:10px 14px;margin:10px 0;border-radius:10px;"
            "border:1px solid rgba(128,128,128,.35);background:rgba(128,128,128,.10)'>"
            f"{body}</div>")

def title(txt):
    return f"<div style='font-size:1.15em;font-weight:bold;margin:14px 0 4px'>{txt}</div>"

def wrap(html):
    return f"<div dir='rtl' style='text-align:right;line-height:1.9'>{html}</div>"

HR = "<hr style='border:none;border-top:1px solid rgba(128,128,128,.4);margin:16px 0'>"

# ---------- الكروت ----------
def profit_line(c, suit):
    if c in ECON:
        lo, hi = profit_range(c, suit)
        return f"💰 <b>الربح التقريبي:</b> {money(lo, hi)} للفدان في الموسم"
    return "💰 <b>الربح:</b> بستان، ربحه على سنين مش موسم واحد"

def grow_card(i, r, g):
    lines = [
        f"<span style='font-size:1.1em'><b>{i}. {AR[r['crop']]}</b></span>",
        f"📉 <b>العجز:</b> مصر كلها ناقصها {p(r['deficit'])} من احتياجها منه",
        f"🌤️ <b>الملاءمة:</b> جو وتربة {g} مناسبين له بنسبة {p(r['suitability'])} ({fit_word(r['suitability'])})",
    ]
    if r["weak"]:
        lines.append(f"⚠️ <b>أضعف نقطة:</b> {r['weak']} مش في أحسن حالاتها له هنا")
    lines.append(profit_line(r["crop"], r["suitability"]))
    return card(*lines)

def export_card(r, g):
    c = r["crop"]
    countries = EXPORT_TO.get(c)
    to = ("فممكن تصدّره لـ: " + "، ".join(countries)) if countries \
         else "ومفيش دول تصدير محددة متسجلة له لسه"
    return card(
        f"<span style='font-size:1.1em'><b>{AR[c]}</b></span>",
        f"📈 <b>الفايض:</b> مصر بتنتج منه أكتر من احتياجها بنسبة {p(r['surplus'])}",
        f"🌤️ <b>الملاءمة:</b> مناسب لـ {g} بنسبة {p(r['suitability'])}",
        f"🌍 <b>التصدير:</b> {to}",
    )

def tips(gov):
    other = [x for x in GOV_AR if x != gov][-1]
    return (HR + "<div style='opacity:.85'>ممكن تسألني: <b>ليه الأول؟</b> · "
            f"<b>إيه اللي أصدّره؟</b> · أو اكتب <b>عدد الفدادين</b> · أو جرّب <b>{GOV_AR[other]}</b></div>")

# ---------- خطة الفدادين (موسمين في السنة) ----------
def season_candidates(df, season):
    c = df[(df["suitability"] >= MIN_FIT) & (df["crop"].isin(ECON))
           & (df["crop"].map(SEASON) == season)]
    return c.sort_values(["score", "suitability"], ascending=False)

def plan_html(df, g, feddan):
    n = 1 if feddan < 3 else 2 if feddan < 6 else 3
    html = title(f"🧑‍🌾 خطة لـ {num(feddan)} فدان في {g}")
    tlo = thi = 0
    for season, label in (("winter", "❄️ الموسم الشتوي"), ("summer", "☀️ الموسم الصيفي")):
        cand = season_candidates(df, season).head(n)
        if cand.empty:
            html += card(f"<b>{label}</b>", "مفيش محصول مناسب كفاية للموسم ده في المحافظة دي.")
            continue
        shares = {1: [1], 2: [0.6, 0.4], 3: [0.5, 0.3, 0.2]}[len(cand)]
        lines = [f"<b>{label}</b>"]
        for (_, r), s in zip(cand.iterrows(), shares):
            area = round(feddan * s, 1)
            lo, hi = profit_range(r["crop"], r["suitability"])
            lo, hi = lo * area, hi * area
            tlo += lo
            thi += hi
            tag = " (سوقه متشبع، مناسب للتصدير)" if r["deficit"] < 0.05 else ""
            lines.append(f"🌱 <b>{AR[r['crop']]}</b>{tag}: {num(area)} فدان، "
                         f"ربح تقريبي {money(lo, hi)}")
        html += card(*lines)
    html += card(f"💰 <b>الإجمالي التقريبي في السنة (الموسمين):</b> {money(tlo, thi)}",
                 "🔁 الأرض بتتزرع مرتين في السنة، شتوي وبعده صيفي.")
    html += ("<div style='opacity:.75;font-size:.9em'>⚠️ الأرباح تقديرات تقريبية، "
             "والفعلي بيتغير حسب السعر والتكاليف والموسم.</div>")
    return html

# ---------- المحادثة ----------
def norm(s):
    s = re.sub(r"[\u064b-\u065f]", "", s)
    for a, b in {"أ": "ا", "إ": "ا", "آ": "ا", "ة": "ه", "ى": "ي"}.items():
        s = s.replace(a, b)
    return s.strip().lower()

GOV_KEYS = {norm(ar): en for en, ar in GOV_AR.items()}
GOV_KEYS.update({en.lower(): en for en in GOV_AR})
state = {"gov": None, "feddan": None}

AR_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

def find_gov(msg):
    m = norm(msg)
    for k, en in GOV_KEYS.items():
        if k in m:
            return en
    return None

def find_num(msg):
    m = re.search(r"\d+(?:\.\d+)?", msg.translate(AR_DIGITS))
    return float(m.group()) if m else None

def chat(message, history):
    m = norm(message)
    gov = find_gov(message)
    if gov:
        state["gov"] = gov
    n = find_num(message)
    if n and 0 < n <= 10000:
        state["feddan"] = n
    gov = state["gov"]

    if gov is None:
        names = "، ".join(GOV_AR.values())
        return wrap(f"أهلاً! 🌱 اكتب اسم محافظتك وأقولك تزرع إيه.{HR}المتاح دلوقتي: <b>{names}</b>")

    df = recommend_v3(gov)
    df["surplus"] = df["crop"].map(lambda c: float(surplus.get(c, 0)))
    g = GOV_AR[gov]
    grow = df[(df["score"] > 0) & (df["suitability"] >= MIN_FIT)].head(3)
    export = df[(df["deficit"] < 0.05) & (df["suitability"] >= 0.5)]

    if any(k in m for k in ["صدر", "تصدير"]):
        if export.empty:
            return wrap(f"في {g} مفيش محصول فايض ومناسب للتصدير حالياً." + tips(gov))
        html = title(f"🚢 اللي تقدر تصدّره من {g}")
        html += "".join(export_card(r, g) for _, r in export.iterrows())
        return wrap(html + tips(gov))

    if any(k in m for k in ["ليه", "لماذا", "سبب"]):
        if grow.empty:
            return wrap(f"مفيش محصول ناقص ومناسب كفاية لـ {g} حالياً." + tips(gov))
        top = grow.iloc[0]
        html = title(f"🏆 ليه {AR[top['crop']]} هو الأول في {g}؟")
        html += card(
            f"📉 مصر كلها ناقصها {p(top['deficit'])} من احتياجها منه",
            f"🌤️ جو وتربة {g} مناسبين له بنسبة {p(top['suitability'])}",
            f"🧮 <b>الدرجة</b> = العجز × الملاءمة = <b><bdi>{top['score']:.2f}</bdi></b>",
        )
        return wrap(html + tips(gov))

    # لو كتب عدد فدادين في الرسالة دي: نعرض الخطة بس
    if n and state["feddan"]:
        return wrap(plan_html(df, g, state["feddan"]) + tips(gov))

    html = title(f"📍 {g}")
    html += HR + title("🌱 ازرع")
    if grow.empty:
        html += "مفيش محصول ناقص ومناسب كفاية للمحافظة دي حالياً."
    else:
        html += "".join(grow_card(i, r, g) for i, (_, r) in enumerate(grow.iterrows(), 1))
    if not export.empty:
        html += HR + title("🚢 فايض ممكن يتصدّر")
        html += "".join(export_card(r, g) for _, r in export.iterrows())

    html += HR
    if state["feddan"]:
        html += plan_html(df, g, state["feddan"])
    else:
        html += ("<b>🧑‍🌾 عندك كام فدان؟</b> اكتب العدد وأنا أقولك تزرع إيه "
                 "وتوزّع الأرض إزاي.")
    return wrap(html + tips(gov))

gr.close_all()
demo = gr.ChatInterface(
    fn=chat,
    chatbot=gr.Chatbot(rtl=True, sanitize_html=False, height=600),
    title="🌱 ارضنا",
    description="اكتب اسم محافظتك وأنا أقولك تزرع إيه، وإيه اللي تصدّره.",
    examples=["المنوفية", "أسوان", "المنيا", "أسيوط"],
    cache_examples=False,
)
demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d0a41b3b11162d752c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
